gpt2-med-big-gensim

In [1]:
!pip install pandarallel transformers sentencepiece wandb


  Preparing metadata (setup.py) ... done
  Created wheel for pandarallel: filename=pandarallel-1.6.5-py3-none-any.whl size=16674 sha256=ca1c0c1450481e94d3c445a025bfeaa287a5c423e9a73e58613bede7a8e61117
  Stored in directory: /root/.cache/pip/wheels/46/f9/0d/40c9cd74a7cb8dc8fe57e8d6c3c19e2c730449c0d3f2bf66b5
Successfully built pandarallel


In [2]:
import os
import random
import nltk
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.optim import AdamW
from tqdm import tqdm
from pandarallel import pandarallel
import warnings
warnings.filterwarnings("ignore")
import wandb
from kaggle_secrets import UserSecretsClient
import numpy as np
import math

# Must be set before anything uses CUDA
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

pandarallel.initialize(progress_bar=False)
nltk.download("cmudict")
from nltk.corpus import cmudict
cmu_dict = cmudict.dict()
print("Setup done!")


INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


[nltk_data] Downloading package cmudict to /usr/share/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


Setup done!


In [3]:
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("NLP_Project_Wandb"))

wandb.init(project="NLP_Project", name="GPT2_med_big_gensim")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhaysharma01702 (abhaysharma01702-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
############################################
# PARAMETERS
############################################

CSV_PATH         = "/kaggle/input/datasets/abhaysharma01702/emotion-lyrics/lyrics_with_emotions.csv"
CHECKPOINT_DIR   = "/kaggle/working/checkpoints"
FINAL_DIR        = "/kaggle/working/gpt2_med_gensim"

EPOCHS           = 3
BATCH_SIZE       = 32       # 16 per GPU across 2 GPUs
MAX_LEN          = 120
LR               = 5e-5
TARGET_PER_CLASS = 60000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)
print("GPUs available:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))


Device: cuda
GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [5]:
############################################
# LOAD & BALANCE DATASET
############################################

df = pd.read_csv(CSV_PATH)
df = df[["cleaned_lyrics", "dominant_emotion"]].dropna()
df.columns = ["text", "emotion"]
df["emotion"] = df["emotion"].str.strip().str.lower()
df = df[df["text"].str.len() > 50]

print("Original distribution:")
print(df["emotion"].value_counts())

parts = []
for emotion, group in df.groupby("emotion"):
    n = min(TARGET_PER_CLASS, len(group))
    parts.append(group.sample(n, random_state=42))

balanced_df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nBalanced distribution:")
print(balanced_df["emotion"].value_counts())
print(f"Total: {len(balanced_df)} samples")


Original distribution:
emotion
happy       1763813
angry        644733
neutral      546585
sad          372004
fear          23883
surprise      16821
Name: count, dtype: int64

Balanced distribution:
emotion
sad         60000
happy       60000
angry       60000
neutral     60000
fear        23883
surprise    16821
Name: count, dtype: int64
Total: 280704 samples


In [6]:
############################################
# FORMAT TEXTS
############################################

texts = balanced_df.apply(
    lambda x: f"<{x['emotion']}> {x['text']}", axis=1
).tolist()

print(f"Sample: {texts[0][:150]}")


Sample: <sad> oh pain
i'm doing bad
i'm gettin' answers to some questions
that i never should have asked
and it's getting old
it's decomposing fact
'cause whe


In [7]:
############################################
# TOKENIZER
############################################

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

emotions = balanced_df["emotion"].unique().tolist()
special_tokens = [f"<{e}>" for e in emotions]
tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

print("Emotion tokens:", special_tokens)
print("Vocab size:", len(tokenizer))


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Emotion tokens: ['<sad>', '<happy>', '<angry>', '<neutral>', '<fear>', '<surprise>']
Vocab size: 50263


In [8]:
############################################
# DATASET — LAZY TOKENIZATION
############################################

class LyricsDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        ids = enc["input_ids"].squeeze()
        return {
            "input_ids":      ids,
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         ids.clone()
        }

dataset = LyricsDataset(texts, tokenizer, MAX_LEN)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

print(f"Dataset: {len(dataset)} samples | Steps/epoch: {len(loader)}")


Dataset: 280704 samples | Steps/epoch: 8772


In [9]:
############################################
# LOAD MODEL
############################################

# Clear any leftover VRAM from previous runs
torch.cuda.empty_cache()

model = GPT2LMHeadModel.from_pretrained("gpt2-medium") 
model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False
model.to(DEVICE)

if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)
    print(f"Using {torch.cuda.device_count()} GPUs")
else:
    print("Single GPU mode")

scaler    = torch.cuda.amp.GradScaler()
optimizer = AdamW(model.parameters(), lr=LR)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {total_params:.0f}M parameters")

for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {alloc:.1f}GB / {total:.1f}GB used")

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Using 2 GPUs
Model: 355M parameters
GPU 0: 1.4GB / 15.6GB used
GPU 1: 0.0GB / 15.6GB used


In [10]:
############################################
# TRAINING
############################################

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

model.train()

for epoch in range(EPOCHS):

    total_loss = 0
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for step, batch in enumerate(pbar):

        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels         = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss.mean()   # mean across GPUs

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        if step % 5000 == 0 and step > 0:
            ckpt = f"{CHECKPOINT_DIR}/step_{step}"
            os.makedirs(ckpt, exist_ok=True)
            raw = model.module if hasattr(model, "module") else model
            raw.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            print(f"\nCheckpoint saved at step {step}")

    avg_loss = total_loss / len(loader)
    perplexity = math.exp(avg_loss)

    wandb.log({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "perplexity": perplexity
    })
    print(f"\nEpoch {epoch+1} avg loss: {avg_loss:.4f}")

    epoch_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}"
    os.makedirs(epoch_path, exist_ok=True)
    raw = model.module if hasattr(model, "module") else model
    raw.save_pretrained(epoch_path)
    tokenizer.save_pretrained(epoch_path)
    print(f"Epoch {epoch+1} saved → {epoch_path}")


Epoch 1/3:  57%|█████▋    | 5000/8772 [1:18:50<59:21,  1.06it/s, loss=3.1224]  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1/3:  57%|█████▋    | 5001/8772 [1:18:52<1:45:19,  1.68s/it, loss=3.1224]


Checkpoint saved at step 5000


Epoch 1/3: 100%|██████████| 8772/8772 [2:18:24<00:00,  1.06it/s, loss=2.9246]  

...


Checkpoint saved at step 5000


Epoch 3/3: 100%|██████████| 8772/8772 [2:18:37<00:00,  1.05it/s, loss=3.0499]  


Epoch 3 avg loss: 2.9973


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3 saved → /kaggle/working/checkpoints/epoch_3


In [11]:
############################################
# SAVE FINAL MODEL
############################################

raw = model.module if hasattr(model, "module") else model
raw.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Final model saved →", FINAL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved → /kaggle/working/gpt2_med_gensim


In [22]:
!pip install -U kaggle

In [21]:
!echo '{"title":"gpt2-med-large-gensim","id":"abhaysharma01702/gpt2-med-large-gensim","licenses":[{"name":"CC0-1.0"}]}' > /kaggle/working/gpt2_med_gensim/dataset-metadata.json
!kaggle datasets create -p /kaggle/working/gpt2_med_gensim

Starting upload for file tokenizer_config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|██████████████████████████████████████████| 421/421 [00:00<00:00, 2.24kB/s]
Upload successful: tokenizer_config.json (421B)
Starting upload for file generation_config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|████████████████████████████████████████████| 118/118 [00:00<00:00, 671B/s]
Upload successful: generation_config.json (118B)
Starting upload for file config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|██████████████████████████████████████| 0.99k/0.99k [00:00<00:00, 5.93kB/s]
Upload successful: config.json (1014B)
Starting upload for file model.safetensors
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|█████████

In [20]:
!kaggle datasets version -p /kaggle/working/gpt2_med_gensim -m "update"

Starting upload for file tokenizer_config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|██████████████████████████████████████████| 421/421 [00:00<00:00, 2.18kB/s]
Upload successful: tokenizer_config.json (421B)
Starting upload for file generation_config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|████████████████████████████████████████████| 118/118 [00:00<00:00, 573B/s]
Upload successful: generation_config.json (118B)
Starting upload for file config.json
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|██████████████████████████████████████| 0.99k/0.99k [00:00<00:00, 5.42kB/s]
Upload successful: config.json (1014B)
Starting upload for file model.safetensors
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'
100%|█████████

In [17]:
# !kaggle datasets version -p /kaggle/working/gpt2_med_gensim -m "fixed slug"

Starting upload for file tokenizer_config.json
100%|██████████████████████████████████████████| 421/421 [00:00<00:00, 2.10kB/s]
Upload successful: tokenizer_config.json (421B)
Starting upload for file generation_config.json
100%|████████████████████████████████████████████| 118/118 [00:00<00:00, 658B/s]
Upload successful: generation_config.json (118B)
Starting upload for file config.json
100%|██████████████████████████████████████| 0.99k/0.99k [00:00<00:00, 5.46kB/s]
Upload successful: config.json (1014B)
Starting upload for file model.safetensors
100%|██████████████████████████████████████| 1.32G/1.32G [00:16<00:00, 83.5MB/s]
Upload successful: model.safetensors (1GB)
Starting upload for file tokenizer.json
100%|██████████████████████████████████████| 3.39M/3.39M [00:00<00:00, 16.8MB/s]
Upload successful: tokenizer.json (3MB)
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion


In [13]:
############################################
# SYLLABLE + RHYME UTILITIES
############################################

vowels = "aeiouy"

def fallback_syllables(word):
    word = word.lower()
    count, prev = 0, False
    for c in word:
        if c in vowels:
            if not prev: count += 1
            prev = True
        else:
            prev = False
    if word.endswith("e"): count -= 1
    return max(1, count)

def syllables(word):
    word = word.lower()
    if word in cmu_dict:
        return max(sum(1 for p in pron if str.isdigit(p[-1])) for pron in cmu_dict[word])
    return fallback_syllables(word)

def rhyme_part(word):
    word = word.lower()
    if word not in cmu_dict: return None
    pron = cmu_dict[word][0]
    for i in range(len(pron)-1, -1, -1):
        if pron[i][-1].isdigit(): return tuple(pron[i:])
    return None

def rhyme_score(line, rhyme_word):
    words = line.split()
    if not words: return 0
    r1, r2 = rhyme_part(words[-1].lower()), rhyme_part(rhyme_word)
    return 1 if (r1 and r2 and r1 == r2) else 0

def meter_score(line, target=8):
    return -abs(sum(syllables(w) for w in line.lower().split()) - target)


In [14]:
############################################
# GENERATION
############################################

BAD_WORDS = {"yeah", "oh", "uh", "ayy", "ooh", "mmm"}

def bad_line(line):
    words = line.lower().split()
    if len(words) < 3: return True
    if len(set(words)) / len(words) < 0.4: return True
    if sum(1 for w in words if w in BAD_WORDS) > len(words) / 2: return True
    return False

def sample_line(emotion, max_len=30):
    raw       = model.module if hasattr(model, "module") else model
    prompt    = f"<{emotion}>"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    output    = raw.generate(
        input_ids,
        max_new_tokens=max_len,
        do_sample=True,
        top_p=0.9,
        temperature=0.85,
        repetition_penalty=1.25,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.replace(prompt, "").strip()

def generate_best_line(emotion, rhyme_word=None, meter=8, candidates=5):
    best_line, best_score = None, -999
    for _ in range(candidates):
        line = sample_line(emotion)
        if not line or bad_line(line): continue
        score = meter_score(line, meter) + (rhyme_score(line, rhyme_word) * 5 if rhyme_word else 0)
        if score > best_score:
            best_score, best_line = score, line
    return best_line or sample_line(emotion)

def generate_song(emotion, meter=8):
    raw = model.module if hasattr(model, "module") else model
    raw.eval()
    with torch.no_grad():
        l1 = generate_best_line(emotion, meter=meter)
        l2 = generate_best_line(emotion, meter=meter)
        l3 = generate_best_line(emotion, rhyme_word=l1.split()[-1], meter=meter)
        l4 = generate_best_line(emotion, rhyme_word=l2.split()[-1], meter=meter)
    return "\n".join([l1, l2, l3, l4])


In [23]:
############################################
# TEST INFERENCE
############################################

for emotion in ["happy", "sad", "angry", "surprise", "fear", "neutral"]:
    print(f"\n{'='*40}")
    print(f"EMOTION: {emotion.upper()}")
    print("="*40)
    print(generate_song(emotion))



EMOTION: HAPPY
hey, hey
hey little baby
hey oh, hey there baby
can't you see
that your love is true?
my heart's on fire, and i'm so fucking hot
i think i'll fuck this city up
oh, and all you people around here
yeah, uh
yeah! yeah
she said she like my style
said that i'm the one for her to hold and ride
i said
i can't help but think it's true
that if you love me
i could never get enough of you, you know?

EMOTION: SAD
i've got no money
no where to go, oh yeah!...oh yeah !
oh yeah!..yeah!!
you are a stranger in my home
and i’m scared to tell you where we’re going, ohhh
you take me
it's too hard
to say that i'm sorry for what's happened
i wish you could see the way that you made me feel, now
it's all in the way
i'll never understand why you do this to me and i don't know what you're trying too
can we

EMOTION: ANGRY
this is your chance to leave
take my hand and leave
there's no one else but you
so don't say i'm wrong, yeah
oh, the words they're coming out
of my mouth
they're falling from

  adding: kaggle/working/gpt2_med_gensim/ (stored 0%)
  adding: kaggle/working/gpt2_med_gensim/model.safetensors (deflated 7%)
  adding: kaggle/working/gpt2_med_gensim/tokenizer.json (deflated 82%)
  adding: kaggle/working/gpt2_med_gensim/generation_config.json (deflated 25%)
  adding: kaggle/working/gpt2_med_gensim/config.json (deflated 53%)
  adding: kaggle/working/gpt2_med_gensim/tokenizer_config.json (deflated 51%)
